# Pipeline Hierárquica com Transformers (DistilBERT) — Submissão 2

**Abordagem:** Dois classificadores em cascata:
1. **Stage 1** — Binário: Human vs AI
2. **Stage 2** — Multi-classe (4 classes AI): Anthropic, Google, Meta, OpenAI

Usa fine-tuning de **DistilBERT** (`distilbert-base-uncased`) para cada estágio.

## 0. Setup — Imports e Configuração

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix
)
from sklearn.model_selection import train_test_split

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device ──
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")

## 1. Hiperparâmetros

In [ ]:
# ── Shared ──
MAX_LEN       = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1

# ── Stage 1: Human vs AI ──
EPOCHS_S1 = 5

# ── Stage 2: AI sub-classes ──
EPOCHS_S2 = 5

# ── Labels ──
LABELS_BINARY   = ['AI', 'Human']
LABELS_AI_CLASS = ['Anthropic', 'Google', 'Meta', 'OpenAI']
LABELS_FINAL    = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

## 2. Carregar e Preparar os Dados

- **Treino:** `dataset_limpo.csv` (4995 amostras, 999 por classe)
- **Teste externo (professor):** `subm1_labels_revealed.csv`
- **Submissão 2:** `subm2.csv`

In [ ]:
# ── Load train data ──
df = pd.read_csv('../data/dataset_limpo.csv', sep=';').dropna(subset=['Text', 'Label'])
print(f"Dataset principal: {len(df)} amostras")
print(df['Label'].value_counts())

# ── Load professor's revealed labels for evaluation ──
df_eval = pd.read_csv('../data/subm1_labels_revealed.csv', sep=';', encoding='utf-8-sig').dropna(subset=['Text', 'Label'])
print(f"\nDataset avaliação (subm1 revelado): {len(df_eval)} amostras")
print(df_eval['Label'].value_counts())

## 3. Preparação dos Dados — Stage 1 (Human vs AI)

In [ ]:
# ── Binary labels ──
df['Label_Bin'] = df['Label'].apply(lambda x: 'Human' if x == 'Human' else 'AI')

# ── Stratified split ──
X_train_s1, X_val_s1, y_train_s1, y_val_s1 = train_test_split(
    df['Text'].to_numpy(), df['Label_Bin'].to_numpy(),
    test_size=0.2, random_state=SEED, stratify=df['Label_Bin'].to_numpy()
)

label2idx_s1 = {label: i for i, label in enumerate(LABELS_BINARY)}
idx2label_s1 = {i: label for label, i in label2idx_s1.items()}

print(f"Train S1: {len(X_train_s1)}  |  Val S1: {len(X_val_s1)}")
print(f"Train dist: AI={sum(y_train_s1=='AI')}, Human={sum(y_train_s1=='Human')}")
print(f"Val   dist: AI={sum(y_val_s1=='AI')}, Human={sum(y_val_s1=='Human')}")

## 4. Preparação dos Dados — Stage 2 (AI Sub-classes)

In [ ]:
# ── Only AI samples ──
df_ai = df[df['Label'] != 'Human'].copy()

X_train_s2, X_val_s2, y_train_s2, y_val_s2 = train_test_split(
    df_ai['Text'].to_numpy(), df_ai['Label'].to_numpy(),
    test_size=0.2, random_state=SEED, stratify=df_ai['Label'].to_numpy()
)

label2idx_s2 = {label: i for i, label in enumerate(LABELS_AI_CLASS)}
idx2label_s2 = {i: label for label, i in label2idx_s2.items()}

print(f"Train S2: {len(X_train_s2)}  |  Val S2: {len(X_val_s2)}")
for lbl in LABELS_AI_CLASS:
    print(f"  {lbl}: train={sum(y_train_s2==lbl)}, val={sum(y_val_s2==lbl)}")

## 5. Dataset PyTorch para DistilBERT

In [ ]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, label2idx, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.label2idx = label2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.label2idx[self.labels[idx]]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long)
        }


class TextInferenceDataset(Dataset):
    """Dataset for inference (no labels)."""
    def __init__(self, texts, tokenizer, max_len=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

## 6. Funções de Treino e Avaliação

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, warmup_ratio,
                weight_decay, device, class_weights=None):
    """Fine-tune a DistilBERT model with optional class weighting."""
    model.to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_loader) * epochs
    warmup_steps = int(total_steps * warmup_ratio)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    if class_weights is not None:
        loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(device))
    else:
        loss_fn = nn.CrossEntropyLoss()

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state = None

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        total_loss, correct, total = 0, 0, 0

        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = loss_fn(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item() * len(labels)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += len(labels)

        train_loss = total_loss / total
        train_acc = correct / total

        # ── Validate ──
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = loss_fn(logits, labels)

                val_loss += loss.item() * len(labels)
                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += len(labels)

        val_loss_avg = val_loss / val_total
        val_acc = val_correct / val_total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss_avg)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        print(f"Epoch {epoch+1}/{epochs} — "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss_avg:.4f}, Acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Restore best model
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n✓ Restaurado melhor modelo (val_acc = {best_val_acc:.4f})")

    return model, history


def predict_probs(model, dataloader, device):
    """Get predicted probabilities for all samples."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)
            all_probs.append(probs.cpu().numpy())
    return np.vstack(all_probs)


def evaluate_model(model, dataloader, device, idx2label, label_names):
    """Evaluate and print metrics."""
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    pred_labels = [idx2label[p] for p in all_preds]
    true_labels = [idx2label[l] for l in all_labels]

    acc = accuracy_score(true_labels, pred_labels)
    bal_acc = balanced_accuracy_score(true_labels, pred_labels)
    print(f"Accuracy: {acc:.4f}  |  Balanced Accuracy: {bal_acc:.4f}")
    print()
    print(classification_report(true_labels, pred_labels, labels=label_names, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(true_labels, pred_labels, labels=label_names))
    return acc, bal_acc

## 7. Inicializar Tokenizer

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")

## 8. Stage 1 — Treino do Classificador Binário (Human vs AI)

Fine-tuning de DistilBERT para distinguir texto humano de texto gerado por IA.
Usamos class weights para equilibrar (1 Human : ~4 AI).

In [ ]:
# ── Datasets ──
train_ds_s1 = TextClassificationDataset(X_train_s1, y_train_s1, tokenizer, label2idx_s1, MAX_LEN)
val_ds_s1   = TextClassificationDataset(X_val_s1, y_val_s1, tokenizer, label2idx_s1, MAX_LEN)

train_loader_s1 = DataLoader(train_ds_s1, batch_size=BATCH_SIZE, shuffle=True)
val_loader_s1   = DataLoader(val_ds_s1,   batch_size=BATCH_SIZE)

# ── Class weights (inverse frequency) ── 
n_ai = sum(y_train_s1 == 'AI')
n_human = sum(y_train_s1 == 'Human')
total_s1 = n_ai + n_human
w_ai = total_s1 / (2 * n_ai)
w_human = total_s1 / (2 * n_human)
class_weights_s1 = [w_ai, w_human]  # order matches LABELS_BINARY = ['AI', 'Human']
print(f"Class weights S1: AI={w_ai:.3f}, Human={w_human:.3f}")

# ── Model ──
model_s1 = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

print(f"\n{'='*60}")
print("TREINO STAGE 1 — Human vs AI")
print(f"{'='*60}")

model_s1, history_s1 = train_model(
    model_s1, train_loader_s1, val_loader_s1,
    epochs=EPOCHS_S1, lr=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    device=DEVICE, class_weights=class_weights_s1
)

### 8.1 Avaliação Stage 1 (Validação Interna)

In [ ]:
print("\n── Stage 1: Validação Interna ──")
acc_s1, bal_s1 = evaluate_model(model_s1, val_loader_s1, DEVICE, idx2label_s1, LABELS_BINARY)

## 9. Stage 2 — Treino do Classificador AI (Anthropic / Google / Meta / OpenAI)

Fine-tuning de DistilBERT apenas com os textos AI para distinguir a origem.

In [ ]:
# ── Datasets ──
train_ds_s2 = TextClassificationDataset(X_train_s2, y_train_s2, tokenizer, label2idx_s2, MAX_LEN)
val_ds_s2   = TextClassificationDataset(X_val_s2, y_val_s2, tokenizer, label2idx_s2, MAX_LEN)

train_loader_s2 = DataLoader(train_ds_s2, batch_size=BATCH_SIZE, shuffle=True)
val_loader_s2   = DataLoader(val_ds_s2,   batch_size=BATCH_SIZE)

# ── Model ──
model_s2 = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=4
)

print(f"\n{'='*60}")
print("TREINO STAGE 2 — AI Sub-classes")
print(f"{'='*60}")

model_s2, history_s2 = train_model(
    model_s2, train_loader_s2, val_loader_s2,
    epochs=EPOCHS_S2, lr=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    device=DEVICE
)

### 9.1 Avaliação Stage 2 (Validação Interna)

In [ ]:
print("\n── Stage 2: Validação Interna ──")
acc_s2, bal_s2 = evaluate_model(model_s2, val_loader_s2, DEVICE, idx2label_s2, LABELS_AI_CLASS)

## 10. Pipeline Combinada — Inferência

Dado um texto:
1. Stage 1 classifica como Human ou AI
2. Se AI → Stage 2 classifica em Anthropic / Google / Meta / OpenAI

In [ ]:
def pipeline_predict(texts, tokenizer, model_s1, model_s2,
                     label2idx_s1, idx2label_s1, idx2label_s2,
                     device, max_len=256, batch_size=16,
                     human_confidence_threshold=0.5):
    """
    Two-stage hierarchical prediction.
    
    Args:
        human_confidence_threshold: minimum confidence to classify as Human.
            If Stage 1 predicts Human but confidence < threshold,
            we fall through to Stage 2 anyway.
    """
    # ── Stage 1: Binary prediction ──
    ds = TextInferenceDataset(texts, tokenizer, max_len)
    loader = DataLoader(ds, batch_size=batch_size)
    
    probs_s1 = predict_probs(model_s1, loader, device)  # (N, 2)
    preds_s1 = probs_s1.argmax(axis=1)
    confs_s1 = probs_s1.max(axis=1)
    
    human_idx_s1 = label2idx_s1['Human']
    
    # ── Stage 2: AI sub-class prediction for all texts ──
    # (We predict for all; we'll only use results for AI-classified texts)
    probs_s2 = predict_probs(model_s2, loader, device)  # (N, 4)
    preds_s2 = probs_s2.argmax(axis=1)
    
    # ── Combine ──
    final_labels = []
    for i in range(len(texts)):
        if preds_s1[i] == human_idx_s1 and confs_s1[i] >= human_confidence_threshold:
            final_labels.append('Human')
        else:
            final_labels.append(idx2label_s2[preds_s2[i]])
    
    return final_labels, probs_s1, probs_s2


print("✓ Pipeline combinada definida")

## 11. Avaliação da Pipeline Completa — Validação Interna

In [ ]:
# ── Reconstruct full validation set ──
# We need texts with their ORIGINAL 5-class labels
df_val_full = df.iloc[
    train_test_split(
        np.arange(len(df)), test_size=0.2, random_state=SEED, stratify=df['Label_Bin'].to_numpy()
    )[1]
]

X_val_full = df_val_full['Text'].to_numpy()
y_val_full = df_val_full['Label'].to_numpy()

preds_val, _, _ = pipeline_predict(
    X_val_full, tokenizer, model_s1, model_s2,
    label2idx_s1, idx2label_s1, idx2label_s2,
    DEVICE, MAX_LEN, BATCH_SIZE
)

acc_pipeline = accuracy_score(y_val_full, preds_val)
bal_acc_pipeline = balanced_accuracy_score(y_val_full, preds_val)

print(f"\n{'='*60}")
print("PIPELINE COMPLETA — Validação Interna (5 classes)")
print(f"{'='*60}")
print(f"Accuracy: {acc_pipeline:.4f}")
print(f"Balanced Accuracy: {bal_acc_pipeline:.4f}")
print()
print(classification_report(y_val_full, preds_val, labels=LABELS_FINAL, zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_val_full, preds_val, labels=LABELS_FINAL))

## 12. Tuning do Threshold de Confiança para Human

In [ ]:
# ── Try different human confidence thresholds ──
best_threshold = 0.5
best_pipeline_acc = acc_pipeline

for th in np.arange(0.30, 0.85, 0.05):
    preds_th, _, _ = pipeline_predict(
        X_val_full, tokenizer, model_s1, model_s2,
        label2idx_s1, idx2label_s1, idx2label_s2,
        DEVICE, MAX_LEN, BATCH_SIZE,
        human_confidence_threshold=th
    )
    acc_th = accuracy_score(y_val_full, preds_th)
    bal_th = balanced_accuracy_score(y_val_full, preds_th)
    print(f"  Threshold={th:.2f}  →  Acc={acc_th:.4f}  Bal.Acc={bal_th:.4f}")
    
    if bal_th > best_pipeline_acc:
        best_pipeline_acc = bal_th
        best_threshold = th

print(f"\n✓ Melhor threshold: {best_threshold:.2f} (Bal.Acc={best_pipeline_acc:.4f})")

## 13. Avaliação no Dataset do Professor (`subm1_labels_revealed.csv`)

Este é o teste externo real do professor.

In [ ]:
X_eval = df_eval['Text'].to_numpy()
y_eval = df_eval['Label'].to_numpy()

preds_eval, probs_s1_eval, probs_s2_eval = pipeline_predict(
    X_eval, tokenizer, model_s1, model_s2,
    label2idx_s1, idx2label_s1, idx2label_s2,
    DEVICE, MAX_LEN, BATCH_SIZE,
    human_confidence_threshold=best_threshold
)

acc_eval = accuracy_score(y_eval, preds_eval)
bal_acc_eval = balanced_accuracy_score(y_eval, preds_eval)

print(f"\n{'='*60}")
print("AVALIAÇÃO EXTERNA — subm1_labels_revealed.csv")
print(f"{'='*60}")
print(f"Accuracy: {acc_eval:.4f}  ({acc_eval*100:.1f}%)")
print(f"Balanced Accuracy: {bal_acc_eval:.4f}")
print(f"\n(Melhor anterior: 61.2%)")
print()
print(classification_report(y_eval, preds_eval, labels=LABELS_FINAL, zero_division=0))
print("\nConfusion Matrix:")
cm = confusion_matrix(y_eval, preds_eval, labels=LABELS_FINAL)
print(pd.DataFrame(cm, index=LABELS_FINAL, columns=LABELS_FINAL))

## 14. Comparação — Modelo Único 5-classes (DistilBERT)

Para comparar, treinamos também um único DistilBERT nas 5 classes directamente.

In [ ]:
# ── Full 5-class split ──
X_train_5c, X_val_5c, y_train_5c, y_val_5c = train_test_split(
    df['Text'].to_numpy(), df['Label'].to_numpy(),
    test_size=0.2, random_state=SEED, stratify=df['Label'].to_numpy()
)

label2idx_5c = {label: i for i, label in enumerate(LABELS_FINAL)}
idx2label_5c = {i: label for label, i in label2idx_5c.items()}

train_ds_5c = TextClassificationDataset(X_train_5c, y_train_5c, tokenizer, label2idx_5c, MAX_LEN)
val_ds_5c   = TextClassificationDataset(X_val_5c, y_val_5c, tokenizer, label2idx_5c, MAX_LEN)

train_loader_5c = DataLoader(train_ds_5c, batch_size=BATCH_SIZE, shuffle=True)
val_loader_5c   = DataLoader(val_ds_5c,   batch_size=BATCH_SIZE)

model_5c = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=5
)

print(f"\n{'='*60}")
print("TREINO MODELO ÚNICO — 5 Classes")
print(f"{'='*60}")

model_5c, history_5c = train_model(
    model_5c, train_loader_5c, val_loader_5c,
    epochs=5, lr=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
    device=DEVICE
)

In [ ]:
# ── Evaluate 5-class model on internal validation ──
print("\n── Modelo 5-classes: Validação Interna ──")
acc_5c, bal_5c = evaluate_model(model_5c, val_loader_5c, DEVICE, idx2label_5c, LABELS_FINAL)

# ── Evaluate on professor's data ──
eval_ds_5c = TextClassificationDataset(X_eval, y_eval, tokenizer, label2idx_5c, MAX_LEN)
eval_loader_5c = DataLoader(eval_ds_5c, batch_size=BATCH_SIZE)

print("\n── Modelo 5-classes: subm1_labels_revealed.csv ──")
acc_5c_eval, bal_5c_eval = evaluate_model(model_5c, eval_loader_5c, DEVICE, idx2label_5c, LABELS_FINAL)

## 15. Comparação de Resultados

In [ ]:
results = pd.DataFrame({
    'Modelo': [
        'Pipeline 2-stage (DistilBERT)',
        'Modelo único 5-class (DistilBERT)',
        'Melhor anterior (subm1)'
    ],
    'Acc Interna': [f"{acc_pipeline*100:.1f}%", f"{acc_5c*100:.1f}%", "—"],
    'Acc Externa (subm1_revealed)': [f"{acc_eval*100:.1f}%", f"{acc_5c_eval*100:.1f}%", "61.2%"],
    'Bal.Acc Externa': [f"{bal_acc_eval*100:.1f}%", f"{bal_5c_eval*100:.1f}%", "—"]
})
print(results.to_string(index=False))

## 16. Selecção do Melhor Modelo e Geração da Submissão 2

In [ ]:
# ── Decide which model to use ──
if acc_eval >= acc_5c_eval:
    print("✓ Pipeline 2-stage é melhor ou igual → usar para submissão")
    USE_PIPELINE = True
else:
    print("✓ Modelo 5-classes é melhor → usar para submissão")
    USE_PIPELINE = False

## 17. Gerar Ficheiro de Submissão 2

In [ ]:
# ── Load blind test set ──
df_subm = pd.read_csv('../subm2.csv', sep=';', encoding='utf-8-sig')
print(f"Submissão 2: {len(df_subm)} textos")
print(df_subm.head())

X_subm = df_subm['Text'].to_numpy()

if USE_PIPELINE:
    preds_subm, _, _ = pipeline_predict(
        X_subm, tokenizer, model_s1, model_s2,
        label2idx_s1, idx2label_s1, idx2label_s2,
        DEVICE, MAX_LEN, BATCH_SIZE,
        human_confidence_threshold=best_threshold
    )
else:
    # Use 5-class model
    subm_ds = TextInferenceDataset(X_subm, tokenizer, MAX_LEN)
    subm_loader = DataLoader(subm_ds, batch_size=BATCH_SIZE)
    probs_subm = predict_probs(model_5c, subm_loader, DEVICE)
    preds_subm = [idx2label_5c[p] for p in probs_subm.argmax(axis=1)]

# ── Build submission (ONLY ID and Label, no Text!) ──
df_out = pd.DataFrame({
    'ID': df_subm['ID'].to_numpy(),
    'Label': preds_subm
})

print(f"\nDistribuição das predições:")
print(pd.Series(preds_subm).value_counts())

# ── Save ──
os.makedirs('../Subm2', exist_ok=True)
output_path = '../Subm2/subm2-g3-MIA-B.csv'
df_out.to_csv(output_path, sep=';', index=False)
print(f"\n✓ Submissão guardada em: {output_path}")
print(df_out.head(10))

## 18. Guardar Modelos

In [ ]:
save_dir = '../models/pytorch_models/saved_models'
os.makedirs(save_dir, exist_ok=True)

# ── Save Stage 1 ──
torch.save(model_s1.state_dict(), os.path.join(save_dir, 'distilbert_stage1_binary.pt'))
print("✓ Stage 1 (Human vs AI) guardado")

# ── Save Stage 2 ──
torch.save(model_s2.state_dict(), os.path.join(save_dir, 'distilbert_stage2_ai_class.pt'))
print("✓ Stage 2 (AI sub-classes) guardado")

# ── Save 5-class ──
torch.save(model_5c.state_dict(), os.path.join(save_dir, 'distilbert_5class.pt'))
print("✓ Modelo 5-classes guardado")

print(f"\nModelos guardados em: {save_dir}")